In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import os
import re
import warnings
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import hstack, coo_matrix

warnings.filterwarnings('ignore')

# Custom SMAPE metric
def smape(y_true, y_pred):
    y_true_unlogged = np.expm1(y_true)
    y_pred_unlogged = np.expm1(y_pred)
    numerator = np.abs(y_pred_unlogged - y_true_unlogged)
    denominator = (np.abs(y_true_unlogged) + np.abs(y_pred_unlogged)) / 2
    smape_val = np.mean(numerator / (denominator + 1e-8)) * 100
    return 'smape', smape_val, False

print("Libraries imported.")

Libraries imported.


In [2]:
# Define paths
PROCESSED_DATA_FOLDER = '../data/processed'
TRAIN_FILE_PATH = os.path.join(PROCESSED_DATA_FOLDER, 'train_processed.parquet')
TEST_FILE_PATH = os.path.join(PROCESSED_DATA_FOLDER, 'test_processed.parquet')

# Load the data
train_df = pd.read_parquet(TRAIN_FILE_PATH)
test_df = pd.read_parquet(TEST_FILE_PATH)

train_df['catalog_content'] = train_df['catalog_content'].astype(str)
test_df['catalog_content'] = test_df['catalog_content'].astype(str)

print(f"Loaded train data: {train_df.shape}")
print(f"Loaded test data: {test_df.shape}")

Loaded train data: (75000, 48)
Loaded test data: (75000, 46)


In [3]:
print("Extracting brand features...")
brand_regex = re.compile(r'Item Name:\s*([\w\’\'\-\.&]+)')
def extract_brand(text_series):
    brands = text_series.str.extract(brand_regex, expand=False).fillna('Unknown').str.lower()
    return brands

train_df['brand'] = extract_brand(train_df['catalog_content'])
test_df['brand'] = extract_brand(test_df['catalog_content'])

brand_encoder = LabelEncoder()
all_brands = pd.concat([train_df['brand'], test_df['brand']])
brand_encoder.fit(all_brands)
train_df['brand_encoded'] = brand_encoder.transform(train_df['brand'])
test_df['brand_encoded'] = brand_encoder.transform(test_df['brand'])
print("Brand extraction and encoding complete.")

# --- NEW FEATURE: unit_measure ---
print("Creating 'unit_measure' feature...")
# We add +1e-6 to avoid division by zero, just in case pack_size is 0
train_df['unit_measure'] = train_df['total_measure'] / (train_df['pack_size'] + 1e-6)
test_df['unit_measure'] = test_df['total_measure'] / (test_df['pack_size'] + 1e-6)

# Fill any new NaNs with 0
train_df['unit_measure'] = train_df['unit_measure'].fillna(0)
test_df['unit_measure'] = test_df['unit_measure'].fillna(0)

print("New feature 'unit_measure' created.")
train_df[['pack_size', 'total_measure', 'unit_measure']].head()

Extracting brand features...
Brand extraction and encoding complete.
Creating 'unit_measure' feature...
New feature 'unit_measure' created.


,pack_size,total_measure,unit_measure
0,6.0,72.00,11.999998
1,4.0,32.00,7.999998
2,6.0,11.40,1.900000
3,1.0,11.25,11.249989
4,1.0,12.00,11.999988


In [4]:
# 80-20 train-validation split
X_train, X_val = train_test_split(train_df, test_size=0.2, random_state=42)
y_train = X_train['log_price']
y_val = X_val['log_price']
print(f"Training data shape: {X_train.shape}")
print(f"Validation data shape: {X_val.shape}")

Training data shape: (60000, 51)
Validation data shape: (15000, 51)


In [6]:
print("Starting TF-IDF vectorization...")
tfidf_vec = TfidfVectorizer(
    ngram_range=(1, 2), 
    max_features=20000, 
    stop_words='english',
    dtype=np.float32
)
X_train_text = tfidf_vec.fit_transform(X_train['catalog_content'])
X_val_text = tfidf_vec.transform(X_val['catalog_content'])
print(f"TF-IDF training matrix shape: {X_train_text.shape}")

Starting TF-IDF vectorization...
TF-IDF training matrix shape: (60000, 20000)


In [7]:
# Get the full list of numerical feature columns
numerical_features = [
    col for col in train_df.columns 
    if col.startswith('unit_') or 
       col in ['pack_size', 'total_measure', 'unit_measure'] # <-- Added 'unit_measure'
]
print(f"Found {len(numerical_features)} numerical features.")
print(numerical_features)

# Get the numerical data
X_train_num = X_train[numerical_features].values.astype(np.float32)
X_val_num = X_val[numerical_features].values.astype(np.float32)

# Get the brand data
X_train_brand = X_train[['brand_encoded']].values.astype(np.float32)
X_val_brand = X_val[['brand_encoded']].values.astype(np.float32)

# --- Combine features ---
print("Combining TF-IDF, numerical, and brand features...")
X_train_final = hstack((
    X_train_text, 
    coo_matrix(X_train_num), 
    coo_matrix(X_train_brand)
))
X_val_final = hstack((
    X_val_text, 
    coo_matrix(X_val_num), 
    coo_matrix(X_val_brand)
))
print(f"Final training feature matrix shape: {X_train_final.shape}")

Found 45 numerical features.
['pack_size', 'total_measure', 'unit_', 'unit_1', 'unit_2', 'unit_bag', 'unit_bottle', 'unit_box', 'unit_count', 'unit_ct', 'unit_each', 'unit_fl', 'unit_fl ounce', 'unit_fl oz', 'unit_fluid ounce', 'unit_fluid ounces', 'unit_foot', 'unit_gram', 'unit_grams', 'unit_jar', 'unit_k', 'unit_kg', 'unit_lb', 'unit_liters', 'unit_ltr', 'unit_mililitro', 'unit_milliliter', 'unit_millilitre', 'unit_ml', 'unit_none', 'unit_ounce', 'unit_ounces', 'unit_oz', 'unit_pack', 'unit_packs', 'unit_paper cupcake liners', 'unit_per package', 'unit_piece', 'unit_pouch', 'unit_pound', 'unit_pounds', 'unit_product_weight', 'unit_sq ft', 'unit_tea bags', 'unit_measure']
Combining TF-IDF, numerical, and brand features...
Final training feature matrix shape: (60000, 20046)


In [9]:
print("Training TUNED LightGBM model...")

# Find the column index for our categorical feature.
categorical_feature_index = X_train_final.shape[1] - 1
print(f"Categorical feature (brand) is at index: {categorical_feature_index}")

# --- NEW TUNED PARAMETERS ---
lgbm_model_tuned = lgb.LGBMRegressor(
    n_estimators=5000,         # More trees
    learning_rate=0.02,        # Slower learning
    num_leaves=31,             # Default, but good to set
    colsample_bytree=0.8,      # Use 80% of features for each tree (prevents overfitting)
    subsample=0.8,             # Use 80% of data for each tree (prevents overfitting)
    random_state=42,
    n_jobs=-1,
    metric='None'
)

# Train the model
lgbm_model_tuned.fit(
    X_train_final, 
    y_train,
    eval_set=[(X_val_final, y_val)],
    eval_metric=smape,
    callbacks=[
        lgb.early_stopping(stopping_rounds=200),  # Increased patience as learning is slower
        lgb.log_evaluation(period=200)
    ]
)

print("\nModel training complete.")
print(f"Best SMAPE Score: {lgbm_model_tuned.best_score_['valid_0']['smape']}")

Training TUNED LightGBM model...
Categorical feature (brand) is at index: 20045
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 2.888380 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1094550
[LightGBM] [Info] Number of data points in the train set: 60000, number of used features: 19579
[LightGBM] [Info] Start training from score 2.740904
Training until validation scores don't improve for 200 rounds
[200]	valid_0's smape: 58.3419
[400]	valid_0's smape: 55.999
[600]	valid_0's smape: 54.9563
[800]	valid_0's smape: 54.2611
[1000]	valid_0's smape: 53.7704
[1200]	valid_0's smape: 53.4003
[1400]	valid_0's smape: 53.0895
[1600]	valid_0's smape: 52.8416
[1800]	valid_0's smape: 52.633
[2000]	valid_0's smape: 52.4675
[2200]	valid_0's smape: 52.3049
[2400]	valid_0's smape: 52.1612
[2600]	valid_0's smape: 52.0274
[2800]	valid_0's smape: 51.9159
[3000]	valid_0's smape: 51.8201
[3200]	valid_0's smape: 51.7333
[340